<a href="https://colab.research.google.com/github/ime-developeradmin-prog/Gemini-Share-Link/blob/main/Waydroid_Infrastructure_Orchestrator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
"""
Waydroid & Ollama Infrastructure Setup Orchestrator
Target Host: Fedora Linux
Container: Waydroid (LineageOS 13 base, GApps, Magisk Delta)
Author: Android Dev Assistant
"""

import os
import sys
import subprocess
import shutil
import pwd
from pathlib import Path

# ==============================================================================
# CONFIGURATION
# ==============================================================================
USER_NAME = "ielliot"
USER_HOME = Path(f"/home/{USER_NAME}")
SCRIPT_DIR = USER_HOME / "waydroid_script"

# Custom images discovered from your tree output
LOCAL_IMAGES_DIR = SCRIPT_DIR / "imgs"
WAYDROID_IMG_DEST = Path("/var/lib/waydroid/images")

OLLAMA_SYSTEMD_DIR = Path("/etc/systemd/system/ollama.service.d")
OLLAMA_OVERRIDE_CONF = OLLAMA_SYSTEMD_DIR / "override.conf"

# ==============================================================================
# UTILITY FUNCTIONS
# ==============================================================================
def print_step(message: str):
    """Prints a styled header for the current step."""
    print(f"\n\033[1;34m[==== {message} ====]\033[0m")

def print_success(message: str):
    print(f"\033[1;32m[SUCCESS] {message}\033[0m")

def print_error(message: str):
    print(f"\033[1;31m[ERROR] {message}\033[0m")

def run_cmd(cmd: list, cwd: Path = None, check: bool = True, env_vars: dict = None):
    """Executes a shell command safely, capturing and routing outputs."""
    print(f"\033[90m> {' '.join(cmd)}\033[0m")

    env = os.environ.copy()
    if env_vars:
        env.update(env_vars)

    try:
        result = subprocess.run(
            cmd,
            cwd=str(cwd) if cwd else None,
            env=env,
            check=check,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )
        return result.stdout
    except subprocess.CalledProcessError as e:
        print_error(f"Command failed with exit code {e.returncode}")
        print_error(f"Output: {e.stdout}")
        sys.exit(1)

def enforce_root():
    """Ensure the script is run with root privileges (sudo)."""
    if os.geteuid() != 0:
        print_error("This orchestration script requires root privileges.")
        print_error("Please re-run using: sudo python3 waydroid_setup.py")
        sys.exit(1)

# ==============================================================================
# MAIN EXECUTION STEPS
# ==============================================================================
def step1_install_dependencies():
    print_step("Step 1: Installing Fedora Dependencies & Binding Kernel Modules")

    # Install dependencies via DNF
    run_cmd(["dnf", "install", "-y", "wl-clipboard", "iptables", "python3-virtualenv"])

    # Load necessary Android kernel modules
    # Waydroid requires binder_linux and sometimes mac80211_hwsim for networking features
    modules = ["binder_linux", "mac80211_hwsim"]
    for mod in modules:
        try:
            run_cmd(["modprobe", mod], check=False)
            print_success(f"Kernel module '{mod}' loaded (or attempted).")
        except Exception:
            pass # modprobe might fail if it's built-in rather than a module

    print_success("Dependencies installed and modules validated.")

def step2_initialize_waydroid():
    print_step("Step 2: Initializing Waydroid & Python Virtual Environment")

    # 2.A: Copy local images into standard Waydroid daemon directory
    print("Preparing local LineageOS 13 images...")
    if not LOCAL_IMAGES_DIR.exists():
        print_error(f"Local images directory missing at {LOCAL_IMAGES_DIR}")
        print_error("Please place 'system.img' and 'vendor.img' there.")
        sys.exit(1)

    WAYDROID_IMG_DEST.mkdir(parents=True, exist_ok=True)

    # Copy and rename to standard waydroid format
    system_img_src = LOCAL_IMAGES_DIR / "system.img"
    vendor_img_src = LOCAL_IMAGES_DIR / "vendor.img"

    if system_img_src.exists() and vendor_img_src.exists():
        shutil.copy(system_img_src, WAYDROID_IMG_DEST / "system.img")
        shutil.copy(vendor_img_src, WAYDROID_IMG_DEST / "vendor.img")
        print_success("Local system and vendor images seeded.")
    else:
        print_error("Missing specific .img files in the local images directory.")
        sys.exit(1)

    # 2.B: Run waydroid init pointing to local configuration
    # The -f flag forces initialization over existing directories
    run_cmd(["waydroid", "init", "-f"])

    # 2.C: Setup the Python virtualenv for the waydroid_script
    if not SCRIPT_DIR.exists():
        print_error(f"waydroid_script repository not found at {SCRIPT_DIR}")
        sys.exit(1)

    venv_dir = SCRIPT_DIR / "venv"
    if not venv_dir.exists():
        print("Creating Python virtual environment...")
        run_cmd(["python3", "-m", "venv", "venv"], cwd=SCRIPT_DIR)

    print("Installing requirements into virtual environment...")
    pip_bin = venv_dir / "bin" / "pip"
    run_cmd([str(pip_bin), "install", "-r", "requirements.txt"], cwd=SCRIPT_DIR)

    print_success("Waydroid initialized and Python venv prepared.")

def step3_patch_container():
    print_step("Step 3: Patching GApps & Magisk Delta into Container Layer")

    python_bin = SCRIPT_DIR / "venv" / "bin" / "python"

    # Run the main.py automation.
    # NOTE: casualsnek's waydroid_script specifically targets Android versions during install.
    # It will manipulate the read-only rootfs of the container.
    print("Executing waydroid_script for GApps and Magisk...")

    try:
        # Install GApps
        run_cmd([str(python_bin), "main.py", "install", "gapps"], cwd=SCRIPT_DIR)

        # Install Magisk (The script handles standard Magisk; ensure it's configured for Delta if you modified the fork)
        run_cmd([str(python_bin), "main.py", "install", "magisk"], cwd=SCRIPT_DIR)

        print_success("GApps and Magisk successfully patched into the Waydroid layer.")
    except Exception as e:
        print_error(f"Failed to apply patches. Please check the waydroid_script logs. Error: {e}")
        sys.exit(1)

def step4_configure_ollama_networking():
    print_step("Step 4: Configuring Ollama Systemd Override & Firewall")

    # 4.A: Systemd override for Ollama Environment Variables
    print("Writing Systemd override configuration for Ollama...")
    OLLAMA_SYSTEMD_DIR.mkdir(parents=True, exist_ok=True)

    override_content = """[Service]
Environment="OLLAMA_HOST=0.0.0.0:11434"
Environment="OLLAMA_ORIGINS=*"
"""
    with open(OLLAMA_OVERRIDE_CONF, "w") as f:
        f.write(override_content)

    # Reload and restart the daemon
    run_cmd(["systemctl", "daemon-reload"])
    run_cmd(["systemctl", "restart", "ollama"])
    print_success("Ollama service restarted with 0.0.0.0 binding and CORS permissive origin.")

    # 4.B: Configure Firewalld for waydroid0 interface
    # Creates a trusted relationship strictly for the container network bridge
    print("Configuring Firewalld for waydroid0 bridge (Port 11434)...")

    # We add the interface to the 'trusted' zone, which allows traffic from the container to the host
    # Or, to be perfectly secure, we isolate it to a custom zone. We'll use specific rules here.
    try:
        # Add interface to trusted to ensure routing works
        run_cmd(["firewall-cmd", "--permanent", "--zone=trusted", "--add-interface=waydroid0"])

        # Explicitly open the port on the host so waydroid can hit 11434
        run_cmd(["firewall-cmd", "--permanent", "--zone=trusted", "--add-port=11434/tcp"])

        # Reload firewall to apply changes
        run_cmd(["firewall-cmd", "--reload"])
        print_success("Firewall configured. waydroid0 can now access host port 11434.")
    except Exception as e:
        print_error(f"Firewalld configuration failed (is firewalld running?). Error: {e}")
        print("You may need to manually run iptables/firewalld commands depending on your Fedora setup.")

def main():
    enforce_root()

    print("\n\033[1;36m*** Starting Android Dev Assistant Environment Provisioning ***\033[0m")

    step1_install_dependencies()
    step2_initialize_waydroid()
    step3_patch_container()
    step4_configure_ollama_networking()

    print("\n\033[1;32m*** Environment Re-establishment Complete! ***\033[0m")
    print("Note: Waydroid requires a session restart to fully mount new layers.")
    print("Run `waydroid session stop` and then launch your UI via the desktop shortcut.")

if __name__ == "__main__":
    main()